# Day 2 — Track B: Llama-3-8B Embeddings on SageMaker

**Certification:** AWS Certified Machine Learning Engineer – Associate (MLA-C01)  
**Domain:** Domain 1 — Data Preparation for ML (24%), Domain 2 — ML Model Development (36%)  
**Instance:** ml.g5.2xlarge (1× NVIDIA A10G, 24 GB VRAM)  
**Region:** us-east-1  
**Estimated Time:** 8–12 minutes  
**Prerequisites:** SageMaker GPU quota approved, HuggingFace token configured

## Learning Objectives

By the end of this notebook, you will:

- Load a 4-bit quantized Llama-3-8B model on GPU
- Generate 4,096-dimensional embeddings for the legal FAQ dataset
- Visualize embeddings with PCA and t-SNE
- Compare Track B (4,096-dim) vs Track A (384-dim) cosine similarity scores
- Save artifacts for Day 4 LoRA fine-tuning

In [ ]:
# Install required packages
%pip install -q transformers bitsandbytes accelerate sentence-transformers scikit-learn matplotlib pandas

In [ ]:
import os
import time
import numpy as np
import pandas as pd
import torch
from datetime import datetime
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from transformers import (
    AutoTokenizer,
    AutoModel,
    BitsAndBytesConfig,
)
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics.pairwise import cosine_similarity

# Paths and configuration
MODEL_NAME: str = "meta-llama/Meta-Llama-3-8B-Instruct"
INSTANCE_TYPE: str = "ml.g5.2xlarge"
GPU_VRAM_GB: int = 24
BATCH_SIZE: int = 4
MAX_LENGTH: int = 512
ARTIFACTS_DIR: Path = Path("artifacts")
ARTIFACTS_DIR.mkdir(exist_ok=True)

print(f"Date      : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Model     : {MODEL_NAME}")
print(f"Instance  : {INSTANCE_TYPE}")
print(f"VRAM      : {GPU_VRAM_GB} GB")
print(f"Batch size: {BATCH_SIZE}")
print(f"Max length: {MAX_LENGTH}")

In [ ]:
# Verify GPU is available and inspect device properties
print("=" * 60)
print("CUDA AVAILABILITY CHECK")
print("=" * 60)

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA not available. This notebook requires a GPU instance "
        "(ml.g5.2xlarge with NVIDIA A10G)."
    )

props = torch.cuda.get_device_properties(0)
print(f"  CUDA available : {torch.cuda.is_available()}")
print(f"  CUDA version   : {torch.version.cuda}")
print(f"  Device name    : {torch.cuda.get_device_name(0)}")
print(f"  Total VRAM     : {props.total_memory / (1024**3):.2f} GB")
print(f"  Multiprocessors: {props.multi_processor_count}")
print(f"  Compute cap    : {props.major}.{props.minor}")

In [ ]:
# Load the 15-entry legal FAQ dataset
print("=" * 60)
print("STEP 1: LOAD FAQ DATASET")
print("=" * 60)

FAQ_DATA: list[dict] = [
    # Contracts (5 entries)
    {"id": 1, "category": "Contracts",
     "question": "What makes a contract legally binding?",
     "answer": "A contract is binding when it includes an offer, acceptance, consideration, and mutual intent to be bound."},
    {"id": 2, "category": "Contracts",
     "question": "Can a contract be oral?",
     "answer": "Yes, oral contracts can be valid, but some agreements must be in writing under the Statute of Frauds."},
    {"id": 3, "category": "Contracts",
     "question": "What happens if one party breaches a contract?",
     "answer": "The non-breaching party may seek remedies such as damages, specific performance, or contract rescission."},
    {"id": 4, "category": "Contracts",
     "question": "What is a force majeure clause?",
     "answer": "It excuses performance due to unforeseeable events beyond the parties' control, such as natural disasters."},
    {"id": 5, "category": "Contracts",
     "question": "How are contract disputes resolved?",
     "answer": "Disputes may be resolved through negotiation, mediation, arbitration, or litigation."},
    # NDAs (5 entries)
    {"id": 6, "category": "NDAs",
     "question": "What is an NDA?",
     "answer": "A Non-Disclosure Agreement is a legal contract that protects confidential information from third parties."},
    {"id": 7, "category": "NDAs",
     "question": "When should an NDA be signed?",
     "answer": "An NDA should be signed before sharing any sensitive or proprietary information with another party."},
    {"id": 8, "category": "NDAs",
     "question": "What information can be protected by an NDA?",
     "answer": "Trade secrets, business plans, customer lists, financial data, and technical designs."},
    {"id": 9, "category": "NDAs",
     "question": "How long does an NDA last?",
     "answer": "Duration varies but commonly ranges from two to five years, or indefinitely for trade secrets."},
    {"id": 10, "category": "NDAs",
     "question": "What are the consequences of violating an NDA?",
     "answer": "Violations can result in injunctions, monetary damages, and reputational harm."},
    # Intellectual Property (5 entries)
    {"id": 11, "category": "Intellectual Property",
     "question": "What are the main types of intellectual property?",
     "answer": "The main types are patents, trademarks, copyrights, and trade secrets."},
    {"id": 12, "category": "Intellectual Property",
     "question": "How long does a copyright last?",
     "answer": "In the U.S., copyright generally lasts for the life of the author plus 70 years."},
    {"id": 13, "category": "Intellectual Property",
     "question": "What does a patent protect?",
     "answer": "A patent protects new, useful, and non-obvious inventions, granting exclusive rights."},
    {"id": 14, "category": "Intellectual Property",
     "question": "What is a trademark?",
     "answer": "A trademark protects words, phrases, symbols, or designs that identify goods or services."},
    {"id": 15, "category": "Intellectual Property",
     "question": "How can a company protect trade secrets?",
     "answer": "Through confidentiality agreements, restricted access, and security measures."},
]

# Build text representations for embedding
texts: list[str] = [
    f"Q: {entry['question']} A: {entry['answer']}"
    for entry in FAQ_DATA
]
categories: list[str] = [entry["category"] for entry in FAQ_DATA]
ids: list[str] = [str(entry["id"]) for entry in FAQ_DATA]

# Report dataset statistics
print(f"  Total entries : {len(texts)}")
for cat in sorted(set(categories)):
    count = categories.count(cat)
    print(f"    {cat:<22}: {count}")
print()
print(f"  Sample FAQ 1 : {texts[0][:90]}...")

In [ ]:
# Load the Llama-3 tokenizer with padding token configured
print("=" * 60)
print("STEP 2: LOAD TOKENIZER")
print("=" * 60)

t0: float = time.time()
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

# Llama-3 does not have a pad token; use EOS token instead
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"  Load time   : {time.time() - t0:.2f}s")
print(f"  Vocab size  : {tokenizer.vocab_size}")
print(f"  Pad token   : {repr(tokenizer.pad_token)} (id={tokenizer.pad_token_id})")
print(f"  EOS token   : {repr(tokenizer.eos_token)} (id={tokenizer.eos_token_id})")
print(f"  BOS token   : {repr(tokenizer.bos_token)} (id={tokenizer.bos_token_id})")

In [ ]:
# Tokenize all 15 FAQ entries with padding and truncation
print("=" * 60)
print("STEP 3: TOKENIZE")
print("=" * 60)

t0 = time.time()
encoded = tokenizer(
    texts,
    padding="max_length",
    truncation=True,
    max_length=MAX_LENGTH,
    return_tensors="pt",
)
print(f"  Tokenization : {time.time() - t0:.2f}s")

print(f"  input_ids      : {tuple(encoded['input_ids'].shape)}")
print(f"  attention_mask : {tuple(encoded['attention_mask'].shape)}")
print(f"  Max length     : {MAX_LENGTH}")

# Show first 15 token IDs for the first 2 entries
print()
print(f"  Sample input_ids[0] (first 15 tokens): {encoded['input_ids'][0][:15].tolist()}")
print(f"  Sample input_ids[1] (first 15 tokens): {encoded['input_ids'][1][:15].tolist()}")

In [ ]:
# Load Llama-3-8B with 4-bit NormalFloat quantization
print("=" * 60)
print("STEP 4: LOAD MODEL (4-bit NF4)")
print("=" * 60)
print("  This loads ~6 GB of model weights into VRAM.")
print("  Expected time: 3–5 minutes.\n")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

print("  BitsAndBytesConfig:")
print(f"    load_in_4bit              : {bnb_config.load_in_4bit}")
print(f"    bnb_4bit_compute_dtype    : {bnb_config.bnb_4bit_compute_dtype}")
print(f"    bnb_4bit_quant_type       : {bnb_config.bnb_4bit_quant_type}")
print(f"    bnb_4bit_use_double_quant : {bnb_config.bnb_4bit_use_double_quant}")

t0 = time.time()
model = AutoModel.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
model.eval()
print()
print(f"  Model loaded in {time.time() - t0:.2f}s")

# VRAM usage after model load
vram_used: float = torch.cuda.memory_allocated() / (1024**3)
vram_reserved: float = torch.cuda.memory_reserved() / (1024**3)
print()
print("  VRAM after load:")
print(f"    Allocated : {vram_used:.2f} GB / {GPU_VRAM_GB} GB ({100 * vram_used / GPU_VRAM_GB:.0f}%)")
print(f"    Reserved  : {vram_reserved:.2f} GB")
print(f"    Hidden dim: {model.config.hidden_size}")

In [ ]:
# Define mean pooling: averages token vectors into document vectors
def mean_pooling(
    last_hidden_state: torch.Tensor,
    attention_mask: torch.Tensor,
) -> torch.Tensor:
    """
    Average token embeddings into a single document vector.

    The attention mask ensures padding tokens are excluded from the average.

    Args:
        last_hidden_state: Shape [batch, seq_len, hidden_dim] — output of the model.
        attention_mask:    Shape [batch, seq_len] — 1 for real tokens, 0 for padding.

    Returns:
        Pooled embeddings of shape [batch, hidden_dim].
    """
    mask = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
    summed = torch.sum(last_hidden_state * mask, dim=1)
    counts = torch.clamp(mask.sum(dim=1), min=1e-9)
    return summed / counts

print("Mean pooling function defined.")
print(f"  Input : last_hidden_state [{BATCH_SIZE}, seq_len, {model.config.hidden_size}]")
print(f"  Output: document_vector  [{BATCH_SIZE}, {model.config.hidden_size}]")

In [ ]:
# Generate embeddings in batches to avoid OOM
print("=" * 60)
print(f"STEP 5: GENERATE EMBEDDINGS (batch_size={BATCH_SIZE})")
print("=" * 60)

input_ids = encoded["input_ids"]
attention_mask = encoded["attention_mask"]
n = input_ids.size(0)
n_batches = (n + BATCH_SIZE - 1) // BATCH_SIZE

print(f"  Samples : {n}")
print(f"  Batches : {n_batches}\n")

all_embeddings: list[np.ndarray] = []

with torch.no_grad():
    for i in range(0, n, BATCH_SIZE):
        batch_num = i // BATCH_SIZE + 1
        batch_ids = input_ids[i : i + BATCH_SIZE].cuda()
        batch_mask = attention_mask[i : i + BATCH_SIZE].cuda()

        t0 = time.time()
        outputs = model(input_ids=batch_ids, attention_mask=batch_mask)
        pooled = mean_pooling(outputs.last_hidden_state, batch_mask)
        pooled = pooled.float().cpu().numpy()
        all_embeddings.append(pooled)

        elapsed = time.time() - t0
        end = min(i + BATCH_SIZE - 1, n - 1)
        print(f"  Batch {batch_num}/{n_batches} | samples {i}–{end} | shape {pooled.shape} | {elapsed:.2f}s")

embeddings: np.ndarray = np.concatenate(all_embeddings, axis=0)
print()
print(f"  Final embeddings shape: {embeddings.shape}")
print(f"  Expected               : ({n}, {model.config.hidden_size})")

In [ ]:
# Visualize embeddings in 2D using PCA + t-SNE
print("=" * 60)
print("STEP 6: PCA + t-SNE VISUALIZATION")
print("=" * 60)

# Detect unique categories dynamically
unique_categories = sorted(set(categories))
print(f"  Categories detected: {unique_categories}")

# PCA: reduce from high-dim to 50 components
pca = PCA(n_components=min(50, embeddings.shape[0]), random_state=42)
pca_result = pca.fit_transform(embeddings)
print(f"  PCA shape           : {pca_result.shape}")
print(f"  Explained variance  : {pca.explained_variance_ratio_[:5]}")

# t-SNE: reduce from 50 to 2 dimensions
perplexity = min(5, embeddings.shape[0] - 1)  # avoid t-SNE error if too few samples
tsne = TSNE(n_components=2, perplexity=perplexity, random_state=42, max_iter=1000)
tsne_result = tsne.fit_transform(pca_result)
print(f"  t-SNE shape         : {tsne_result.shape}")

# Plot: one color per category
fig, ax = plt.subplots(figsize=(10, 7))
colors = plt.cm.tab10(np.linspace(0, 1, len(unique_categories)))

for cat_index, cat in enumerate(unique_categories):
    idx = [j for j, c in enumerate(categories) if c == cat]
    ax.scatter(
        tsne_result[idx, 0],
        tsne_result[idx, 1],
        c=[colors[cat_index]],
        label=cat,
        s=140,
        alpha=0.85,
        edgecolors="black",
        linewidths=0.8,
    )
    for sample_idx in idx:
        ax.annotate(
            ids[sample_idx],
            (tsne_result[sample_idx, 0], tsne_result[sample_idx, 1]),
            textcoords="offset points",
            xytext=(6, 6),
            fontsize=8,
            fontweight="bold",
        )

ax.set_title(
    "Track B — Llama-3-8B Embeddings (t-SNE)\n"
    "Legal FAQ: Contracts vs NDAs vs Intellectual Property",
    fontsize=12,
    fontweight="bold",
)
ax.set_xlabel("t-SNE Dimension 1")
ax.set_ylabel("t-SNE Dimension 2")
ax.legend(title="Category", loc="best")
ax.grid(True, alpha=0.3)
plt.tight_layout()

plot_path = ARTIFACTS_DIR / "tsne_plot_trackB.png"
plt.savefig(str(plot_path), dpi=150)
plt.show()
print(f"  Plot saved to: {plot_path}")

In [ ]:
# Measure embedding quality via within-category cosine similarity
print("=" * 60)
print("STEP 7: WITHIN-CATEGORY COSINE SIMILARITY")
print("=" * 60)

track_b_results: dict[str, float] = {}

for cat in unique_categories:
    idx = [j for j, c in enumerate(categories) if c == cat]
    cat_emb = embeddings[idx]

    if len(idx) >= 2:
        sim_matrix = cosine_similarity(cat_emb)
        upper = np.triu_indices(len(idx), k=1)
        sims = sim_matrix[upper]
        mean_sim = float(np.mean(sims))
        track_b_results[cat] = mean_sim
        print(f"  {cat:<22}: {mean_sim:.4f}  ({len(sims)} pairs)")
    else:
        track_b_results[cat] = 0.0
        print(f"  {cat:<22}: N/A (only {len(idx)} sample)")

# Overall average
overall_b = float(np.mean(list(track_b_results.values())))
track_b_results["Overall"] = overall_b
print()
print(f"  {'Overall':<22}: {overall_b:.4f}")

In [ ]:
# Compute Track A baseline with all-MiniLM-L6-v2 (384-dim)
print("=" * 60)
print("STEP 8: TRACK A BASELINE (MiniLM 384-dim)")
print("=" * 60)

from sentence_transformers import SentenceTransformer

st_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
emb_a: np.ndarray = st_model.encode(texts, convert_to_numpy=True)
print(f"  Track A embeddings shape: {emb_a.shape}")

track_a_results: dict[str, float] = {}
for cat in unique_categories:
    idx = [j for j, c in enumerate(categories) if c == cat]
    cat_emb = emb_a[idx]

    if len(idx) >= 2:
        sim_matrix = cosine_similarity(cat_emb)
        upper = np.triu_indices(len(idx), k=1)
        sims = sim_matrix[upper]
        track_a_results[cat] = float(np.mean(sims))
    else:
        track_a_results[cat] = 0.0

track_a_results["Overall"] = float(np.mean(list(track_a_results.values())))
print("  Track A baseline computed.")

In [ ]:
# Side-by-side comparison of Track A vs Track B
print("=" * 60)
print("STEP 9: TRACK A vs TRACK B COMPARISON")
print("=" * 60)

print(f"  {'Category':<22} {'Track A (384D)':<16} {'Track B (4096D)':<16} {'Delta':<10}")
print(f"  {'-'*22} {'-'*16} {'-'*16} {'-'*10}")

for cat in unique_categories + ["Overall"]:
    a = track_a_results.get(cat, 0.0)
    b = track_b_results.get(cat, 0.0)
    delta = b - a
    print(f"  {cat:<22} {a:<16.4f} {b:<16.4f} {delta:<+10.4f}")

print()
print("  Track A: sentence-transformers/all-MiniLM-L6-v2 (384-dim, ~90 MB, 6 layers)")
print("  Track B: meta-llama/Meta-Llama-3-8B-Instruct (4096-dim, 4-bit NF4, ~6 GB VRAM, 32 layers)")

In [ ]:
# Summary of all results for this session
print("=" * 60)
print("FINAL RESULTS SUMMARY")
print("=" * 60)

vram_used = torch.cuda.memory_allocated() / (1024**3)
vram_reserved = torch.cuda.memory_reserved() / (1024**3)

print(f"  Model            : {MODEL_NAME}")
print(f"  Instance         : {INSTANCE_TYPE}")
print("  Quantization     : 4-bit NF4 with double quant")
print(f"  VRAM allocated   : {vram_used:.2f} GB / {GPU_VRAM_GB} GB ({100 * vram_used / GPU_VRAM_GB:.0f}%)")
print(f"  VRAM reserved    : {vram_reserved:.2f} GB")
print(f"  Embedding shape  : {embeddings.shape}")
print(f"  Embedding dim    : {embeddings.shape[1]}")
print(f"  Number of samples: {embeddings.shape[0]}")

print()
print(f"  Within-category cosine similarity:")
print(f"    {'Category':<22} {'Track A':<12} {'Track B':<12}")
print(f"    {'-'*22} {'-'*12} {'-'*12}")
for cat in unique_categories + ["Overall"]:
    a = track_a_results.get(cat, 0)
    b = track_b_results.get(cat, 0)
    print(f"    {cat:<22} {a:<12.4f} {b:<12.4f}")

print()
print(f"  Artifacts saved to: {ARTIFACTS_DIR}/")
print(f"    - embeddings_trackB.npy  ({embeddings.shape[1]}-dim)")
print(f"    - embeddings_trackA.npy  ({emb_a.shape[1]}-dim)")
print("    - metadata.csv")
print("    - tsne_plot_trackB.png")

print()
print(f"  {'='*60}")
print("  Day 2 Track B complete. Ready for Day 4 LoRA fine-tuning.")
print(f"  {'='*60}")